# Predicción de Quiebra Empresarial con Aprendizaje de Máquina
### Dataset: Taiwanese Bankruptcy Prediction (UCI ID: 572)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/REPO/blob/main/Proyecto_Final_Quiebra_Taiwan.ipynb)

- **Integrantes:**
  - Wanerge Almanza Velasquez
  - Jhon Fredy Hoyos Cardenas
  - Miguel Angel Castiblanco Florez
- **Materia:** Modelos y Simulación de Sistemas II
- **Institución:** Universidad de Antioquia (UdeA) - Medellín, Colombia
- **Docente:** Julián D. Arias-Londoño

---
### Estructura del Trabajo
El presente notebook implementa el ciclo completo de desarrollo en Machine Learning acorde a la **Guía del Proyecto Final** y a la teoría vista durante el curso:
1. **Configuración inicial y Registro de Decisiones Metodológicas**
2. **Descripción del Problema y Análisis Exploratorio de Datos (EDA) [10%]**
3. **Estado del Arte y Justificación de Métricas [10%]**
4. **Entrenamiento y Evaluación de Modelos (5 familias obligatorias) [15%]**
5. **Reducción de Dimensionalidad (Análisis univariado, PCA y UMAP) [15%]**
6. **Modelos Complementarios y Explicabilidad (XAI con SHAP)**
7. **Discusión General y Conclusiones Comparativas**


## 1. Configuración del Entorno y Carga de Datos

> **¿Qué hacemos y por qué?**
> Instalamos las dependencias requeridas en caso de ejecutar en Google Colab (`ucimlrepo`, `imbalanced-learn`, `umap-learn`, `shap`), fijamos la semilla de reproducibilidad (`SEED = 42`) y definimos una función de carga dual: si el archivo `data/data.csv` existe localmente se carga directo; de lo contrario, se descarga de forma automatizada desde el repositorio oficial UCI.


In [1]:
# Instalación de librerías para entorno Google Colab (se ejecuta silenciosamente)
!pip install -q ucimlrepo imbalanced-learn umap-learn shap

import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración gráfica global
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = False

# Semilla global de reproducibilidad (enseñada a lo largo de los laboratorios del curso)
SEED = 42
np.random.seed(SEED)

print("Entorno inicializado correctamente con SEED =", SEED)


In [2]:
# Función de carga dual: local o repositorio oficial UCI
def cargar_dataset():
    ruta_local = os.path.join("data", "data.csv")
    if os.path.exists(ruta_local):
        print(f"[INFO] Cargando dataset desde almacenamiento local: {ruta_local}")
        df = pd.read_csv(ruta_local)
    else:
        print("[INFO] Archivo local no encontrado. Descargando automáticamente de UCI Machine Learning Repository (ID 572)...")
        from ucimlrepo import fetch_ucirepo
        taiwan_data = fetch_ucirepo(id=572)
        X = taiwan_data.data.features
        y = taiwan_data.data.targets
        df = pd.concat([y, X], axis=1)
        print("[INFO] Descarga completada con éxito desde UCI.")
        
    # Limpieza de espacios en blanco en los nombres de las columnas
    df.columns = [c.strip() for c in df.columns]
    
    # Normalización de caracteres especiales (por ejemplo el símbolo de moneda Yuan ¥)
    rename_cols = {}
    for col in df.columns:
        if "Yuan" in col:
            rename_cols[col] = col.replace("", "¥").replace("?", "")
    df.rename(columns=rename_cols, inplace=True)
    return df

df_raw = cargar_dataset()
print(f"Dimensiones del conjunto de datos: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")


## 2. Descripción del Problema y Análisis Exploratorio de Datos (10%)

### 2.1 Contexto del Problema y Justificación de Machine Learning
La insolvencia financiera o bancarrota empresarial es un fenómeno crítico con repercusiones macroeconómicas severas en el crédito, el empleo y los mercados bursátiles. La detección anticipada de señales de estrés financiero permite a instituciones financieras, inversores y reguladores mitigar riesgos de impago.

Históricamente, el análisis de bancarrota se abordó con modelos paramétricos lineales clásicos como el **Z-Score de Altman (1968)** (basado en Análisis Discriminante Lineal - LDA) o el **O-Score de Ohlson (1980)** (basado en Regresión Logística). Sin embargo, dichos enfoques adolecen de severas limitaciones en escenarios corporativos reales:
1. **Supuestos distributivos restrictivos:** Exigen normalidad multivariada y homocedasticidad, premisas violadas por los ratios contables que presentan colas pesadas y asimetrías extremas.
2. **Incapacidad ante relaciones no lineales:** La quiebra emerge de interacciones complejas entre endeudamiento, rotación de activos y liquidez a corto plazo.
3. **Asimetría extrema de costos de error:** Clasificar erróneamente a una empresa en quiebra como solvente (Falso Negativo) ocasiona pérdidas crediticias devastadoras frente al costo de investigar una empresa solvente sospechosa (Falso Positivo).

Por tanto, el empleo de **Machine Learning supervisado** se justifica plenamente al permitir modelar fronteras de decisión no lineales complejas sin requerir supuestos distributivos restrictivos, integrando esquemas de regularización y remuestreo para gestionar el desbalance y la asimetría de costos.


In [3]:
# Verificación de integridad: datos faltantes, tipos de datos y duplicados
print("--- AUDITORÍA DE CALIDAD DE DATOS ---")
print(f"Valores nulos en todo el dataset: {df_raw.isna().sum().sum()}")
print(f"Filas duplicadas: {df_raw.duplicated().sum()}")
print(f"Tipos de datos detectados:
{df_raw.dtypes.value_counts()}")


### 2.2 Auditoría de Características y Detección de Columnas Constantes

> **Decisión Metodológica (Filtro preliminar):**
> Al inspeccionar la cantidad de valores únicos por columna, se detecta que la columna `Net Income Flag` posee exactamente **un único valor** en las 6,819 observaciones (todos los valores son 1). Una variable con varianza cero no aporta ninguna información discriminativa a ningún algoritmo y debe ser eliminada del espacio de predictores. Por otro lado, la variable binaria `Liability-Assets Flag` posee solo 8 ocurrencias con valor 1 (de las cuales 6 derivaron en quiebra), actuando como una señal poco frecuente pero de alta sospecha de insolvencia.


In [4]:
# Análisis de valores únicos y eliminación de columna constante
unique_counts = df_raw.nunique()
const_cols = unique_counts[unique_counts <= 1].index.tolist()
print(f"Columnas con varianza cero (constantes): {const_cols}")

# Eliminar columna constante
df = df_raw.drop(columns=const_cols).copy()
print(f"Columnas restantes para entrenamiento: {df.shape[1] - 1} predictores + 1 variable objetivo ('Bankrupt?')")


### 2.3 Análisis de Desbalance Severo de Clases

> **¿Qué observamos?**
> La variable objetivo `Bankrupt?` presenta 6,599 observaciones de empresas solventes (96.77%) y solo 220 registros de empresas en quiebra (3.23%). Esto representa una relación de desbalance cercana a **30 a 1**.
> 
> **Consecuencia metodológica crucial:** La métrica convencional de **Exactitud (Accuracy)** resulta totalmente engañosa e inadmisible como criterio de selección; un clasificador trivial que prediga siempre clase 0 obtendría un 96.77% de accuracy, pero tendría un 0% de sensibilidad en la detección de empresas en quiebra.


In [5]:
# Visualización del desbalance de clases
clase_0 = (df['Bankrupt?'] == 0).sum()
clase_1 = (df['Bankrupt?'] == 1).sum()
n_total = len(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ['#2b5c8f', '#d9534f']

# 1. Gráfico de barras
bars = axes[0].bar(['Solvente (0)', 'Quiebra (1)'], [clase_0, clase_1], color=colors, width=0.5, edgecolor='black')
axes[0].set_title('Figura 1a: Frecuencia Absoluta de Empresas por Clase', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Número de Observaciones', fontsize=10)
for bar in bars:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 120, f'{int(yval):,}
({yval/n_total*100:.2f}%)', 
                 ha='center', va='bottom', fontsize=9.5, fontweight='bold')
axes[0].set_ylim(0, 7500)

# 2. Gráfico de dona
axes[1].pie([clase_0, clase_1], labels=['Solvente (96.77%)', 'Quiebra (3.23%)'], 
            autopct='%1.2f%%', startangle=30, colors=colors, explode=(0, 0.15),
            textprops={'fontsize': 10, 'fontweight': 'bold'}, wedgeprops=dict(width=0.6, edgecolor='black'))
axes[1].set_title('Figura 1b: Proporción Relativa de Clases (Desbalance 30:1)', fontsize=11, fontweight='bold')

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig(os.path.join("figures", "fig1_desbalance_clases.png"), dpi=300, bbox_inches='tight')
plt.show()


### 2.4 Análisis de Escala Extrema, Asimetría y Outliers Contables

> **¿Qué observamos?**
> A diferencia de lo reportado a menudo en la literatura informal, este dataset **no tiene todas las variables normalizadas entre [0, 1]**. 
> Se comprueba que **24 columnas** presentan valores que superan la unidad, alcanzando magnitudes del orden de $10^9$ y $10^{10}$ (como `Cash Turnover Rate`, `Quick Assets/Current Liability`, `Total debt/Total net worth` y tasas de rotación de inventario). Asimismo, **62 de las 94 variables predictoras** presentan un coeficiente de asimetría absoluta ($|skew| > 5$).
> 
> **Decisión Metodológica:**
> Un escalado convencional mediante `StandardScaler` (que depende de la media y la desviación estándar) se distorsionará fuertemente por estos valores extremos. Por ello, para los modelos sensibles a la escala (KNN, Regresión Logística, MLP, SVM) se adoptará un **`QuantileTransformer` con salida normal**, el cual mapea las variables a una distribución gaussiana estándar robusta ante outliers extremos sin perder orden de rango.


In [6]:
# Detección de variables con rangos extremos fuera de [0, 1]
cols_pred = [c for c in df.columns if c != "Bankrupt?"]
mm = df[cols_pred].agg(["min", "max"]).T
out_unit = mm[(mm["min"] < 0) | (mm["max"] > 1)]
print(f"Total de características con valores fuera de [0, 1]: {len(out_unit)}")
print("
Top 8 características con mayores valores máximos observados:")
print(out_unit.sort_values(by="max", ascending=False).head(8))

# Inspección visual de outliers mediante diagramas de caja (Boxplots)
sample_outlier_cols = [
    'Cash Turnover Rate',
    'Quick Assets/Current Liability',
    'Total debt/Total net worth',
    'Net Value Growth Rate'
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for idx, col in enumerate(sample_outlier_cols):
    ax = axes[idx // 2, idx % 2]
    sns.boxplot(data=df, x='Bankrupt?', y=col, ax=ax, palette=['#2b5c8f', '#d9534f'], showmeans=True,
                meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"})
    ax.set_title(f'{col}
(Máx: {df[col].max():.2e} | Skew: {df[col].skew():.2f})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Estado (0: Solvente, 1: Quiebra)', fontsize=9)
    ax.set_ylabel('Valor de Ratio Contable', fontsize=9)

plt.suptitle('Figura 2: Distribución de Ratios Financieros con Escala Extrema y Outliers', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join("figures", "fig2_asimetria_outliers.png"), dpi=300, bbox_inches='tight')
plt.show()


### 2.5 Análisis de Redundancia y Multicolinealidad

> **¿Qué observamos?**
> En contabilidad y finanzas, numerosos ratios miden variantes muy similares de la rentabilidad sobre activos (ROA), liquidez corriente y valor en libros. 
> El análisis de correlación de Pearson sobre las 94 variables predictoras revela **28 pares de características con correlación extrema $|r| \ge 0.90$**, incluyendo 12 pares con $|r| \ge 0.99$ (por ejemplo, variantes de `Operating Gross Margin`, `Net Value Per Share` y `Current Liabilities/Equity`).
> 
> **Decisión Metodológica:**
> Esta alta multicolinealidad motiva formalmente el estudio de **Reducción de Dimensionalidad** en la Sección 5 (mediante filtros univariados, PCA lineal y variedades no lineales con UMAP) para verificar si es posible condensar el espacio sin degradar la capacidad predictiva.


In [7]:
# Cálculo de correlaciones entre variables predictoras
corr_pred = df[cols_pred].corr().abs()
iu = np.triu_indices_from(corr_pred, k=1)
pairs_df = pd.DataFrame({
    "Variable 1": corr_pred.columns[iu[0]],
    "Variable 2": corr_pred.columns[iu[1]],
    "Correlacion (|r|)": corr_pred.values[iu]
})

print(f"Pares con correlación |r| >= 0.90: {(pairs_df['Correlacion (|r|)'] >= 0.90).sum()}")
print(f"Pares con correlación |r| >= 0.95: {(pairs_df['Correlacion (|r|)'] >= 0.95).sum()}")
print(f"Pares con correlación |r| >= 0.99: {(pairs_df['Correlacion (|r|)'] >= 0.99).sum()}")

print("
Top 6 pares con mayor redundancia:")
print(pairs_df.sort_values(by="Correlacion (|r|)", ascending=False).head(6).to_string(index=False))

# Matriz de correlación de las 18 variables más correlacionadas con la quiebra
corr_con_quiebra = df.corr()['Bankrupt?'].abs().sort_values(ascending=False)
top_18_vars = corr_con_quiebra.index[1:19]

plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(df[top_18_vars].corr(), dtype=bool))
sns.heatmap(df[top_18_vars].corr(), mask=mask, cmap='vlag', vmin=-1, vmax=1, annot=True, fmt='.2f', 
            annot_kws={'size': 7}, cbar_kws={'label': 'Coeficiente de Pearson (r)'}, square=True)
plt.title('Figura 3: Matriz de Correlación de las Top 18 Variables más Asociadas a Quiebra', fontsize=11, fontweight='bold', pad=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join("figures", "fig3_matriz_correlaciones.png"), dpi=300, bbox_inches='tight')
plt.show()


### 2.6 Definición y Justificación del Paradigma de Aprendizaje

A partir del análisis exploratorio se establece formalmente el paradigma de modelado:
- **Paradigma:** Aprendizaje Supervisado.
- **Tipo de Problema:** Clasificación Binaria Desbalanceada con Asimetría de Costos ($y_i \in \{0, 1\}$).
- **Justificación:**
  1. Contamos con un conjunto auditado con etiquetas verdaderas proporcionadas por la bolsa de valores (TWSE) a lo largo del periodo 1999-2009.
  2. Los enfoques no supervisados (como agrupamiento o clustering k-means) son inapropiados como modelo de predicción de insolvencia, ya que las empresas en quiebra no conforman un conglomerado denso aislado en el espacio euclidiano, sino instancias periféricas y solapadas en fronteras ruidosas con respecto a las solventes.
  3. El objetivo operativo del sistema es **minimizar el riesgo empírico ponderado por costos**, priorizando la sensibilidad (Recall de quiebras) sin generar una proliferación inmanejable de falsas alarmas.


## 3. Estado del Arte y Lecciones Metodológicas (10%)

El estudio de la insolvencia empresarial sobre el conjunto de datos de Taiwán ha sido abordado profusamente en la literatura internacional. Se seleccionaron cuatro investigaciones clave indexadas en revistas Q1/Q2 para orientar el diseño experimental y prevenir vicios metodológicos comunes:

1. **Liang et al. (2016) - *European Journal of Operational Research* (Elsevier, Q1):**
   - Evaluaron un esquema supervisado binario emparejado (239 quebradas vs 239 solventes) comparando SVM, KNN, CART, MLP y Naive Bayes con técnicas de selección por filtros (SDA, SLR, t-test) y envolturas (GA, RFE). Demostraron que SVM lineal superó a los demás modelos alcanzando un 81.3% de exactitud y minimizando el costo asimétrico de mala clasificación.
   - *Lección crítica:* Al evaluar sobre una muestra balanceada artificialmente (1:1), sus métricas de exactitud no son directamente comparables con la prevalencia real (30:1).

2. **Wang & Liu (2021) - *PLOS ONE* (Q1):**
   - Abordaron el desbalance natural (3.2%) evaluando técnicas de submuestreo (Tomek Links, ENN, RENN, OSS, NCR, Cluster Centroids) sobre nueve familias de modelos.
   - *Aporte fundamental:* Justificaron el uso del **$F_2$-Score** ($\beta=2$) para penalizar cuatro veces más los falsos negativos que los falsos positivos debido a las asimetrías de pérdidas bancarias:
   $$F_2 = 5 \times \frac{\text{Precision} \times \text{Recall}}{4 \times \text{Precision} + \text{Recall}}$$
   - Su mejor configuración fue Gaussian Naive Bayes tras Edited Nearest Neighbors (ENN) con $F_2 = 0.423$, seguido de LDA con $F_2 = 0.368$.

3. **Pham et al. (2025) - *International Journal of Technology* (Q2):**
   - Plantearon un flujo híbrido combinando SMOTE-ENN con optimización por enjambre de partículas binarias (BPSO) para reducir de 75 a 34 variables antes de entrenar SVM, Random Forest y ANN. Reportaron un $F_1$-score de 98.5% con ANN y 97.8% con Random Forest.
   - *Advertencia de fuga de datos (Data Leakage):* La literatura especializada y las buenas prácticas enseñadas en el curso advierten que resultados tan elevados en este dataset suelen deberse a aplicar el sobremuestreo antes de la partición de prueba o evaluar en pliegues sobremuestreados. Nuestro proyecto evaluará estrictamente sobre datos con la prevalencia real.

4. **Azad et al. (2026) - *Discover Artificial Intelligence* (Springer Nature):**
   - Propusieron un marco de destilación de conocimiento supervisado (APEX-DEFT) que comprime un ensamble de Stacking (LightGBM, XGBoost, Bayesian Ridge) en un modelo liviano de *Histogram Gradient Boosting* regularizado con LASSO e Información Mutua.
   - Evaluaron con validación cruzada estratificada de 5 pliegues y métricas robustas como el Coeficiente de Correlación de Matthews (MCC):
   $$\text{MCC} = \frac{TP \times TN - FP \times FN}{\sqrt{(TP + FP)(TP + FN)(TN + FP)(TN + FN)}}$$
   - El modelo retuvo un desempeño competitivo ($F_1 = 69.05\%$, Recall = $81.40\%$, ROC-AUC = $93.17\%$) reduciendo el tiempo de inferencia a 1.55 segundos e incorporando explicabilidad mediante SHAP.


In [8]:
# Matriz Comparativa del Estado del Arte
tabla_sota = pd.DataFrame([
    {
        "Criterio": "Revista / Editorial",
        "Liang et al. (2016)": "EJOR (Elsevier, Q1)",
        "Wang & Liu (2021)": "PLOS ONE (Q1)",
        "Pham et al. (2025)": "Int. J. Technology (Q2)",
        "Azad et al. (2026)": "Discover AI (Springer)"
    },
    {
        "Criterio": "Paradigma / Muestreo",
        "Liang et al. (2016)": "Supervisado emparejado (1:1)",
        "Wang & Liu (2021)": "Submuestreo (ENN, NCR, TL)",
        "Pham et al. (2025)": "SMOTE-ENN + BPSO (34 vars)",
        "Azad et al. (2026)": "Destilación Stacking -> HistGB"
    },
    {
        "Criterio": "Modelos Evaluados",
        "Liang et al. (2016)": "SVM, KNN, CART, MLP, NB",
        "Wang & Liu (2021)": "NB, LDA, NN, XGB, RF, SVM",
        "Pham et al. (2025)": "SVM, Random Forest, ANN",
        "Azad et al. (2026)": "HistGB, LightGBM, XGBoost, MLP"
    },
    {
        "Criterio": "Validación Empleada",
        "Liang et al. (2016)": "10-Fold CV (muestra 478 casos)",
        "Wang & Liu (2021)": "Partición aleatoria (5 réplicas)",
        "Pham et al. (2025)": "Train/Test split 80/20",
        "Azad et al. (2026)": "Stratified 5-Fold CV (prevalencia real)"
    },
    {
        "Criterio": "Métricas Clave",
        "Liang et al. (2016)": "Accuracy, Error I/II, Costo MC",
        "Wang & Liu (2021)": "F2-Score, Precision, Recall",
        "Pham et al. (2025)": "Accuracy, Precision, Recall, F1",
        "Azad et al. (2026)": "F1, Recall, ROC-AUC, MCC, Kappa"
    },
    {
        "Criterio": "Mejor Resultado Reportado",
        "Liang et al. (2016)": "SVM + SDA (Acc = 81.3%)",
        "Wang & Liu (2021)": "NB + ENN (F2 = 0.423)",
        "Pham et al. (2025)": "ANN + SMOTE-ENN (F1 = 98.5%)*",
        "Azad et al. (2026)": "HistGB Destilado (F1 = 69.1%, Rec = 81.4%)"
    }
])

pd.set_option('display.max_columns', None)
tabla_sota


### 3.1 Definición y Justificación Teórica de las Métricas de Evaluación

En concordancia con lo enseñado en la **Clase 05 (Métricas de Error)** y las recomendaciones de la literatura, en problemas con desbalance severo (30:1) se descarta la Exactitud y se fundamenta formalmente el uso de:

1. **Sensibilidad / Recall ($TPR$):**
   $$Recall = \frac{TP}{TP + FN}$$
   Mide la fracción de empresas quebradas que el sistema detecta con éxito. Maximizarla es el objetivo central para evitar la catástrofe financiera de no anticipar una bancarrota.

2. **Precisión ($PPV$):**
   $$Precision = \frac{TP}{TP + FP}$$
   Fracción de alertas emitidas por el modelo que corresponden efectivamente a insolvencias reales.

3. **Área Bajo la Curva Precision-Recall (PR-AUC / Average Precision):**
   $$\text{PR-AUC} = \sum_n (R_n - R_{n-1}) P_n$$
   A diferencia del ROC-AUC (que puede presentar valores artificialmente altos debido a la enorme cantidad de Verdaderos Negativos $TN$), la curva Precision-Recall se enfoca exclusivamente en la clase minoritaria, siendo la métrica más confiable para clasificar y comparar modelos independientes del umbral.

4. **$F_2$-Score ($\beta = 2$):**
   Asigna el doble de importancia al Recall que a la Precisión. Se usará como **criterio objetivo de calibración del umbral de decisión operativo**.

5. **Coeficiente de Correlación de Matthews (MCC):**
   Mide la correlación entre las predicciones binarias y los valores reales considerando las cuatro celdas de la matriz de confusión. Oscila entre -1 y +1, donde 0 indica una predicción aleatoria.


## 4. Entrenamiento y Evaluación de los Modelos (15%)

### 4.1 Configuración Experimental y Protocolo de Validación

> **¿Qué hacemos y por qué?**
> De acuerdo con lo enseñado en la **Clase 06 (Complejidad de modelos, sobreajuste y metodologías de validación)**:
> 1. Realizamos una partición estratificada **Train / Test (70% / 30%)**. El conjunto de test (2,046 muestras, 66 quiebras) se **congela estrictamente** y no se utiliza durante la selección de hiperparámetros ni calibración de umbrales para garantizar una evaluación final completamente imparcial.
> 2. Sobre el conjunto de entrenamiento (4,773 muestras, 154 quiebras), se implementa una **Validación Cruzada Estratificada Repetida (5 pliegues $\times$ 3 repeticiones = 15 evaluaciones)** mediante `RepeatedStratifiedKFold`.
> 3. **Pipeline contra Fuga de Información (Data Leakage):** Todo preprocesamiento (escalado con `QuantileTransformer`, reducciones o remuestreos) se encapsula dentro de un `Pipeline` de scikit-learn/imblearn, de modo que cada pliegue se ajusta (`fit`) única y exclusivamente con los datos de entrenamiento del pliegue.
> 4. **Estimación de Intervalos de Confianza (IC):**
>    - En validación interna: Se calcula el estimador empírico del curso $\text{Media} \pm 2 \cdot \text{std}$ entre los 15 pliegues de validación.
>    - En el conjunto de test: Se estima un **Intervalo de Confianza al 95% mediante Bootstrap no paramétrico estratificado** con 1,000 remuestreos.


In [9]:
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.preprocessing import QuantileTransformer, RobustScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, fbeta_score,
    matthews_corrcoef, balanced_accuracy_score, recall_score, precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)

# 1. Separación de matriz de características X y vector de etiquetas y
X_df = df.drop(columns=['Bankrupt?'])
feature_names = X_df.columns.tolist()
X = X_df.values
y = df['Bankrupt?'].values

# 2. Partición Train / Test estratificada (70% Train, 30% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)

print(f"Dimensiones de Entrenamiento (Train): {X_train.shape[0]} muestras ({y_train.sum()} quiebras, {y_train.mean()*100:.2f}%)")
print(f"Dimensiones de Prueba (Test congelado): {X_test.shape[0]} muestras ({y_test.sum()} quiebras, {y_test.mean()*100:.2f}%)")

# 3. Configuración del protocolo de validación cruzada interna
N_SPLITS = 5
N_REPEATS = 3
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
print(f"Validación interna configurada: RepeatedStratifiedKFold ({N_SPLITS} folds x {N_REPEATS} repeticiones = {N_SPLITS*N_REPEATS} iteraciones)")


In [10]:
# Funciones auxiliares para calibración de umbral y estimación de intervalos de confianza

def find_best_threshold(y_true, y_probs, metric_func=lambda yt, yp: fbeta_score(yt, yp, beta=2, zero_division=0)):
    # Busca el umbral de decisión en [0.05, 0.95] que maximiza el F2-score sobre predicciones OOF
    thresholds = np.linspace(0.05, 0.95, 91)
    scores = [metric_func(y_true, (y_probs >= th).astype(int)) for th in thresholds]
    best_idx = np.argmax(scores)
    return float(thresholds[best_idx]), float(scores[best_idx])

def bootstrap_metrics(y_true, y_probs, threshold=0.5, n_bootstraps=1000, seed=SEED):
    # Calcula Intervalos de Confianza al 95% para test mediante remuestreo bootstrap estratificado
    rng = np.random.RandomState(seed)
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    
    boot_pr_auc = []
    boot_roc_auc = []
    boot_f2 = []
    boot_f1 = []
    boot_mcc = []
    boot_rec = []
    boot_prec = []
    
    for _ in range(n_bootstraps):
        b_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        b_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        b_idx = np.concatenate([b_pos, b_neg])
        
        yb = y_true[b_idx]
        pb = y_probs[b_idx]
        pred_b = (pb >= threshold).astype(int)
        
        boot_pr_auc.append(average_precision_score(yb, pb))
        boot_roc_auc.append(roc_auc_score(yb, pb))
        boot_f2.append(fbeta_score(yb, pred_b, beta=2, zero_division=0))
        boot_f1.append(f1_score(yb, pred_b, zero_division=0))
        boot_mcc.append(matthews_corrcoef(yb, pred_b))
        boot_rec.append(recall_score(yb, pred_b, zero_division=0))
        boot_prec.append(precision_score(yb, pred_b, zero_division=0))
        
    ci = lambda arr: (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))
    return {
        "pr_auc_ci": ci(boot_pr_auc),
        "roc_auc_ci": ci(boot_roc_auc),
        "f2_ci": ci(boot_f2),
        "f1_ci": ci(boot_f1),
        "mcc_ci": ci(boot_mcc),
        "recall_ci": ci(boot_rec),
        "precision_ci": ci(boot_prec)
    }

print("Funciones de evaluación y bootstrap configuradas.")


### 4.2 Entrenamiento y Ajuste de los 5 Modelos de Aprendizaje Obligatorios

De acuerdo con las directrices de la guía del proyecto, se comparan las 5 familias de algoritmos vistas en el curso:
1. **Modelo Paramétrico:**
   - *Regresión Logística:* Modelo discriminativo lineal con ponderación `class_weight='balanced'`, regularización $L_2$ ($C=0.1$) para controlar colinealidad.
   - *Linear Discriminant Analysis (LDA):* Modelo generativo gaussiano con regularización por contracción (`shrinkage='auto'`) para estabilizar la estimación de la covarianza común ante 94 variables altamente correlacionadas.
2. **Modelo No Paramétrico:**
   - *K-Nearest Neighbors (KNN):* Clasificador de vecindario ($k=15$) con ponderación inversa por distancia (`weights='distance'`) para compensar la dispersión en alta dimensión.
3. **Ensamble de Árboles de Decisión:**
   - *Random Forest:* 300 árboles (`n_estimators=300`), profundidad máxima controlada (`max_depth=12`), muestras mínimas en hojas (`min_samples_leaf=2`) y muestreo balanceado por árbol (`class_weight='balanced_subsample'`).
4. **Red Neuronal Artificial (ANN):**
   - *Multi-Layer Perceptron (MLP):* Dos capas ocultas (64 y 32 neuronas), activación ReLU, optimizador Adam, regularización $L_2$ ($\alpha=0.01$), parada temprana (`early_stopping=True`) con paciencia de 15 épocas.
5. **Máquinas de Vectores de Soporte (SVM):**
   - *SVC con Kernel RBF:* Parámetro de penalización $C=1.0$, $\gamma='scale'$, `class_weight='balanced'` y calibración de probabilidades sigmoidales para cálculo de PR-AUC.


In [11]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

# Definición de la batería de modelos base obligatorios
scaler = QuantileTransformer(output_distribution="normal", n_quantiles=200, random_state=SEED)

modelos_base = {
    "Regresion Logistica": LogisticRegression(class_weight="balanced", C=0.1, max_iter=2000, random_state=SEED),
    "LDA (Shrinkage)": LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
    "KNN (k=15, distance)": KNeighborsClassifier(n_neighbors=15, weights="distance"),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=2,
                                           class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
    "Red Neuronal MLP": MLPClassifier(hidden_layer_sizes=(64, 32), alpha=0.01, max_iter=400,
                                      early_stopping=True, n_iter_no_change=15, random_state=SEED),
    "SVM (RBF Kernel)": SVC(kernel="rbf", C=1.0, class_weight="balanced", probability=True, random_state=SEED)
}

resultados_base = []
modelos_ajustados = {}
probabilidades_test = {}
predicciones_test = {}

print("Iniciando validación cruzada y evaluación en Test...")
for nombre, estimador in modelos_base.items():
    t_inicio = time.time()
    pipeline_m = Pipeline([("scaler", scaler), ("model", estimador)])
    
    # 1. Validación Cruzada OOF sobre Train
    oof_probs = np.zeros(len(y_train))
    cv_pr_aucs = []
    cv_roc_aucs = []
    
    for tr_idx, va_idx in cv.split(X_train, y_train):
        pipeline_m.fit(X_train[tr_idx], y_train[tr_idx])
        probs_va = pipeline_m.predict_proba(X_train[va_idx])[:, 1]
        cv_pr_aucs.append(average_precision_score(y_train[va_idx], probs_va))
        cv_roc_aucs.append(roc_auc_score(y_train[va_idx], probs_va))
        oof_probs[va_idx] += probs_va / N_REPEATS

    # 2. Calibración del umbral óptimo (F2) usando OOF de Train
    umbral_opt, f2_oof = find_best_threshold(y_train, oof_probs)
    
    # 3. Ajuste del modelo completo en todo Train
    pipeline_m.fit(X_train, y_train)
    modelos_ajustados[nombre] = pipeline_m
    
    # 4. Evaluación imparcial en Test congelado
    probs_te = pipeline_m.predict_proba(X_test)[:, 1]
    preds_te = (probs_te >= umbral_opt).astype(int)
    
    probabilidades_test[nombre] = probs_te
    predicciones_test[nombre] = preds_te
    
    # 5. Métricas de Test y Bootstrap CI
    pr_auc_te = average_precision_score(y_test, probs_te)
    roc_auc_te = roc_auc_score(y_test, probs_te)
    f2_te = fbeta_score(y_test, preds_te, beta=2, zero_division=0)
    f1_te = f1_score(y_test, preds_te, zero_division=0)
    mcc_te = matthews_corrcoef(y_test, preds_te)
    rec_te = recall_score(y_test, preds_te, zero_division=0)
    prec_te = precision_score(y_test, preds_te, zero_division=0)
    boot_ci = bootstrap_metrics(y_test, probs_te, threshold=umbral_opt)
    
    t_transcurrido = time.time() - t_inicio
    print(f"[{t_transcurrido:4.1f}s] {nombre:<24} | Val PR-AUC: {np.mean(cv_pr_aucs):.3f}±{2*np.std(cv_pr_aucs):.3f} | Test PR-AUC: {pr_auc_te:.3f} | Test F2: {f2_te:.3f} (th={umbral_opt:.2f})")
    
    resultados_base.append({
        "Modelo": nombre,
        "Val PR-AUC (±2σ)": f"{np.mean(cv_pr_aucs):.3f} ± {2*np.std(cv_pr_aucs):.3f}",
        "Val ROC-AUC (±2σ)": f"{np.mean(cv_roc_aucs):.3f} ± {2*np.std(cv_roc_aucs):.3f}",
        "Umbral": umbral_opt,
        "Test PR-AUC": round(pr_auc_te, 4),
        "Test PR-AUC 95% CI": f"[{boot_ci['pr_auc_ci'][0]:.3f}, {boot_ci['pr_auc_ci'][1]:.3f}]",
        "Test ROC-AUC": round(roc_auc_te, 4),
        "Test ROC-AUC 95% CI": f"[{boot_ci['roc_auc_ci'][0]:.3f}, {boot_ci['roc_auc_ci'][1]:.3f}]",
        "Test F2-Score": round(f2_te, 4),
        "Test F2 95% CI": f"[{boot_ci['f2_ci'][0]:.3f}, {boot_ci['f2_ci'][1]:.3f}]",
        "Test F1-Score": round(f1_te, 4),
        "Test MCC": round(mcc_te, 4),
        "Test Recall": round(rec_te, 4),
        "Test Precision": round(prec_te, 4),
        "Tiempo (s)": round(t_transcurrido, 1)
    })

df_res_base = pd.DataFrame(resultados_base)
df_res_base.to_csv(os.path.join("results", "tabla_modelos_base.csv"), index=False)
df_res_base[["Modelo", "Val PR-AUC (±2σ)", "Test PR-AUC", "Test PR-AUC 95% CI", "Test F2-Score", "Test F2 95% CI", "Test Recall", "Test Precision"]]


In [12]:
# Visualización de Curvas ROC y Precision-Recall para todos los modelos base
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
colores = ['#1f77b4', '#aec7e8', '#ff7f0e', '#2ca02c', '#9467bd', '#d62728']

# 1. Curva ROC
for idx, (nombre, probs) in enumerate(probabilidades_test.items()):
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)
    axes[0].plot(fpr, tpr, label=f"{nombre} (AUC={auc_val:.3f})", color=colores[idx % len(colores)], lw=1.8)

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.7, label='Línea Aleatoria (AUC=0.50)')
axes[0].set_title('Figura 4a: Curvas ROC en Test (Espacio Original)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontsize=10)
axes[0].set_ylabel('Sensibilidad / Recall', fontsize=10)
axes[0].legend(loc='lower right', fontsize=8.5)

# 2. Curva Precision-Recall
linea_base_pr = y_test.sum() / len(y_test)
for idx, (nombre, probs) in enumerate(probabilidades_test.items()):
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap_val = average_precision_score(y_test, probs)
    axes[1].plot(rec, prec, label=f"{nombre} (PR-AUC={ap_val:.3f})", color=colores[idx % len(colores)], lw=1.8)

axes[1].axhline(y=linea_base_pr, color='k', linestyle='--', lw=1, alpha=0.7, label=f'Azar (PR={linea_base_pr:.3f})')
axes[1].set_title('Figura 4b: Curvas Precision-Recall en Test (Espacio Original)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Sensibilidad / Recall', fontsize=10)
axes[1].set_ylabel('Precisión', fontsize=10)
axes[1].legend(loc='upper right', fontsize=8.5)

plt.tight_layout()
plt.savefig(os.path.join("figures", "fig4_curvas_roc_pr_base.png"), dpi=300, bbox_inches='tight')
plt.show()


In [13]:
# Visualización de Matrices de Confusión al Umbral Calibrado
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for idx, (nombre, preds) in enumerate(predicciones_test.items()):
    ax = axes[idx // 3, idx % 3]
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['Solvente', 'Quiebra'], yticklabels=['Solvente', 'Quiebra'],
                annot_kws={'size': 11, 'weight': 'bold'})
    f2_val = fbeta_score(y_test, preds, beta=2, zero_division=0)
    rec_val = recall_score(y_test, preds, zero_division=0)
    ax.set_title(f"{nombre}\nF2={f2_val:.3f} | Recall={rec_val:.3f}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicción', fontsize=9)
    ax.set_ylabel('Real', fontsize=9)

plt.suptitle('Figura 5: Matrices de Confusión al Umbral Óptimo en Test (Espacio Original)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join("figures", "fig5_matrices_confusion_base.png"), dpi=300, bbox_inches='tight')
plt.show()


### 4.3 Selección Justificada de los Dos Mejores Modelos

> **Decisión Metodológica (Selección para Reducción de Dimensionalidad):**
> Al analizar de forma conjunta la capacidad de discriminación en validación cruzada y en el conjunto de prueba independiente:
> 1. **Random Forest (Ensamble de Árboles):** Obtuvo el mejor desempeño global en validación ($\text{PR-AUC} = 0.393 \pm 0.123$) y en test ($\text{ROC-AUC} = 0.941$, $F_2 = 0.536$). Su mecanismo de ensamble con particiones aleatorias le otorga una alta robustez ante las distribuciones multimodales y asimétricas de los ratios contables.
> 2. **SVM con Kernel RBF (Máquina de Vectores de Soporte):** Logró un desempeño muy consistente ($\text{Val PR-AUC} = 0.383 \pm 0.163$, $\text{Test PR-AUC} = 0.418$, $\text{Test ROC-AUC} = 0.938$). Al proyectar mediante el kernel gaussiano, logra aislar las empresas insolventes fronterizas maximizando el margen funcional.
> 
> *Modelos descartados para la fase de reducción:*
> - **KNN:** Aunque tras calibración alcanzó buen recall, presenta la peor capacidad de ordenamiento probabilístico ($\text{Val PR-AUC} = 0.316$), sufriendo intensamente la dispersión de distancias euclidianas en 94 dimensiones.
> - **Regresión Logística y LDA:** Tienen la ventaja de la simplicidad e interpretabilidad analítica, pero se ven limitados por la frontera lineal en comparación con los clasificadores no lineales.
> 
> **Modelos seleccionados para evaluar en PCA y UMAP:** **Random Forest** y **SVM (RBF Kernel)**.


## 5. Reducción de Dimensionalidad (15%)

En esta sección se evalúan técnicas de selección y extracción de características para determinar si es posible simplificar la complejidad del modelo, mitigar la multicolinealidad observada y reducir el costo computacional sin deteriorar la capacidad predictiva.

### 5.1 Análisis Individual de Variables (Fase de Filtros)

> **¿Qué hacemos y por qué?**
> Siguiendo lo enseñado en la **Clase 18 (Selección de Características)** y **Clase 21 (Análisis Discriminante de Fisher)**, evaluamos individualmente cada una de las 94 variables predictoras calculadas **estrictamente sobre el conjunto de entrenamiento (Train)** mediante tres criterios estadísticos:
> 1. **Razón de Discriminación de Fisher ($J$):**
>    $$J = \frac{(\mu_1 - \mu_0)^2}{\sigma_1^2 + \sigma_0^2}$$
>    Cuantifica la separación entre las medias de empresas solventes y en quiebra normalizada por la dispersión intraclase.
> 2. **AUC Univariado:** Evalúa la capacidad individual de cada variable como clasificador unidimensional.
> 3. **Correlación Punto-Biserial ($r$):** Mide la asociación lineal entre el ratio contable y la probabilidad de bancarrota.


In [14]:
# Cálculo del análisis univariado sobre Train
X_tr_df = pd.DataFrame(X_train, columns=feature_names)

univariate_data = []
for col in feature_names:
    vals = X_tr_df[col].values
    
    # 1. Correlación punto-biserial
    r_val = float(np.corrcoef(vals, y_train)[0, 1])
    
    # 2. AUC Univariado
    try:
        auc_val = float(roc_auc_score(y_train, vals))
        if auc_val < 0.5:
            auc_val = 1.0 - auc_val
    except:
        auc_val = 0.5
        
    # 3. Razón de Fisher
    x0 = vals[y_train == 0]
    x1 = vals[y_train == 1]
    denom = np.var(x0) + np.var(x1)
    fisher_val = float(((np.mean(x1) - np.mean(x0)) ** 2) / (denom + 1e-12))
    
    univariate_data.append({
        "Variable": col,
        "Fisher Ratio": fisher_val,
        "Univariate AUC": auc_val,
        "Correlación (r)": r_val,
        "|r|": abs(r_val)
    })

df_univariate = pd.DataFrame(univariate_data).sort_values(by="Fisher Ratio", ascending=False)
df_univariate.to_csv(os.path.join("results", "analisis_individual_variables.csv"), index=False)

print("Top 10 Variables más discriminativas según Razón de Fisher:")
df_univariate[["Variable", "Fisher Ratio", "Univariate AUC", "Correlación (r)"]].head(10)


### Identificación de Variables Candidatas a Eliminación por Redundancia

> **Decisión Metodológica (Regla de Poda por Multicolinealidad):**
> Para pares de variables que presenten una correlación extrema $|r| \ge 0.95$ en entrenamiento, se identifica la variable con menor poder discriminativo (menor Razón de Fisher) como **candidata a eliminación**. Este análisis identifica **20 variables redundantes** que pueden descartarse sin pérdida sustantiva de información contable.


In [15]:
# Detección de candidatas a eliminación por redundancia extrema
corr_tr = X_tr_df.corr().abs()
candidatas_eliminar = set()

for i in range(len(feature_names)):
    for j in range(i+1, len(feature_names)):
        if corr_tr.iloc[i, j] >= 0.95:
            var_a = feature_names[i]
            var_b = feature_names[j]
            score_a = df_univariate.loc[df_univariate["Variable"] == var_a, "Fisher Ratio"].values[0]
            score_b = df_univariate.loc[df_univariate["Variable"] == var_b, "Fisher Ratio"].values[0]
            # Descartar la de menor Razón de Fisher
            if score_a >= score_b:
                candidatas_eliminar.add(var_b)
            else:
                candidatas_eliminar.add(var_a)

df_candidatas = pd.DataFrame(sorted(list(candidatas_eliminar)), columns=["Variable Candidata a Eliminar"])
df_candidatas.to_csv(os.path.join("results", "variables_candidatas_eliminar.csv"), index=False)
print(f"Total de variables candidatas a eliminación por redundancia (|r| >= 0.95): {len(df_candidatas)}")
df_candidatas.head(10)


### 5.2 Extracción Lineal de Características con PCA

> **¿Qué hacemos y por qué?**
> Siguiendo lo enseñado en la **Clase 20 (Análisis de Componentes Principales)**:
> 1. Ajustamos PCA sobre los datos escalados de entrenamiento para descomponer la matriz de covarianza en autovectores ortogonales ordenados por varianza explicada.
> 2. **Criterio de Selección de Componentes:** Se evalúan dos umbrales formales de varianza acumulada:
>    - Retención del **90% de la varianza:** Requiere **$k = 23$ componentes** (reducción del **75.5%** de dimensionalidad).
>    - Retención del **95% de la varianza:** Requiere **$k = 31$ componentes** (reducción del **67.0%** de dimensionalidad).
> 3. **Reajuste de Hiperparámetros:** En concordancia explícita con la guía del proyecto (*"cuando se cambia el espacio de características se debe realizar un nuevo ajuste de hiperparámetros"*), se reentrenan y calibran los dos mejores modelos (**Random Forest** y **SVM RBF**) en el nuevo subespacio latente.


In [16]:
from sklearn.decomposition import PCA

# 1. Ajuste de PCA en datos de entrenamiento escalados
scaler_pca = QuantileTransformer(output_distribution="normal", n_quantiles=200, random_state=SEED)
X_train_scaled = scaler_pca.fit_transform(X_train)
X_test_scaled = scaler_pca.transform(X_test)

pca_exploratorio = PCA(random_state=SEED)
pca_exploratorio.fit(X_train_scaled)
var_acum = np.cumsum(pca_exploratorio.explained_variance_ratio_)

k_90 = int(np.argmax(var_acum >= 0.90) + 1)
k_95 = int(np.argmax(var_acum >= 0.95) + 1)

print(f"Número de componentes para 90% de varianza: k = {k_90} (Reducción: {100*(1 - k_90/X.shape[1]):.1f}%)")
print(f"Número de componentes para 95% de varianza: k = {k_95} (Reducción: {100*(1 - k_95/X.shape[1]):.1f}%)")

# 2. Gráfico de Varianza Explicada Acumulada
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(1, len(var_acum)+1), var_acum, 'b-o', markersize=3, label='Varianza Explicada Acumulada')
ax.axhline(y=0.90, color='r', linestyle='--', label=f'90% Varianza (k={k_90})')
ax.axhline(y=0.95, color='g', linestyle='--', label=f'95% Varianza (k={k_95})')
ax.axvline(x=k_90, color='r', linestyle=':', alpha=0.7)
ax.axvline(x=k_95, color='g', linestyle=':', alpha=0.7)
ax.set_title('Figura 6: Curva de Varianza Explicada Acumulada por Componentes Principales (PCA)', fontsize=11, fontweight='bold')
ax.set_xlabel('Número de Componentes Principales (k)', fontsize=10)
ax.set_ylabel('Varianza Acumulada', fontsize=10)
ax.set_ylim(0.2, 1.02)
ax.legend(loc='lower right', fontsize=9.5)
plt.tight_layout()
plt.savefig(os.path.join("figures", "fig6_pca_varianza_explicada.png"), dpi=300, bbox_inches='tight')
plt.show()


In [17]:
# Reentrenamiento de los dos mejores modelos en los espacios PCA (k=23 y k=31)
modelos_pca_eval = {
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=2,
                                           class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
    "SVM (RBF Kernel)": SVC(kernel="rbf", C=1.5, gamma="scale", class_weight="balanced", probability=True, random_state=SEED)
}

resultados_pca = []
for k_comp, etiqueta_var in [(k_90, "90% Var"), (k_95, "95% Var")]:
    pct_red = 100 * (1 - k_comp / X.shape[1])
    for m_nombre, m_clf in modelos_pca_eval.items():
        pipe_pca = Pipeline([
            ("scaler", QuantileTransformer(output_distribution="normal", n_quantiles=200, random_state=SEED)),
            ("pca", PCA(n_components=k_comp, random_state=SEED)),
            ("model", m_clf)
        ])
        
        # Validación interna para umbral OOF
        oof_probs_pca = np.zeros(len(y_train))
        cv_aucs_pca = []
        for tr_i, va_i in cv.split(X_train, y_train):
            pipe_pca.fit(X_train[tr_i], y_train[tr_i])
            p_val = pipe_pca.predict_proba(X_train[va_i])[:, 1]
            cv_aucs_pca.append(average_precision_score(y_train[va_i], p_val))
            oof_probs_pca[va_i] += p_val / N_REPEATS
            
        th_opt_pca, _ = find_best_threshold(y_train, oof_probs_pca)
        
        # Ajuste en todo Train y evaluación en Test
        pipe_pca.fit(X_train, y_train)
        p_te_pca = pipe_pca.predict_proba(X_test)[:, 1]
        pred_te_pca = (p_te_pca >= th_opt_pca).astype(int)
        
        resultados_pca.append({
            "Espacio": f"PCA {etiqueta_var} (k={k_comp})",
            "Reducción (%)": f"{pct_red:.1f}%",
            "Modelo": m_nombre,
            "Val PR-AUC": f"{np.mean(cv_aucs_pca):.3f} ± {2*np.std(cv_aucs_pca):.3f}",
            "Test PR-AUC": round(average_precision_score(y_test, p_te_pca), 4),
            "Test ROC-AUC": round(roc_auc_score(y_test, p_te_pca), 4),
            "Test F2": round(fbeta_score(y_test, pred_te_pca, beta=2, zero_division=0), 4),
            "Test F1": round(f1_score(y_test, pred_te_pca, zero_division=0), 4),
            "Test MCC": round(matthews_corrcoef(y_test, pred_te_pca), 4),
            "Test Recall": round(recall_score(y_test, pred_te_pca, zero_division=0), 4),
            "Test Precision": round(precision_score(y_test, pred_te_pca, zero_division=0), 4)
        })

df_res_pca = pd.DataFrame(resultados_pca)
df_res_pca.to_csv(os.path.join("results", "tabla_pca_resultados.csv"), index=False)
df_res_pca[["Espacio", "Reducción (%)", "Modelo", "Val PR-AUC", "Test PR-AUC", "Test ROC-AUC", "Test F2", "Test Recall", "Test Precision"]]
